# HW3 Part 2




**Course Code:**

**Group number:**

**Student Name:**

**Student ID:**

# Turn Continuity Classification


**Task**: Binary classification — predict whether a spoken turn is **Complete (1)** or **Incomplete (0)**.

**Metric**: Macro-F1 Score.

| Label | Meaning |
|---|---|
| 1 | **Complete** — semantic intent is finished; system can respond |
| 0 | **Incomplete** — intent is unfinished; system should keep listening |


## 0. Environment Setup & Data Loading

In [19]:
!pip install datasets xgboost imbalanced-learn nltk -q

In [20]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

# Core ML
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import classification_report, f1_score
from sklearn.utils import resample
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix

# Advanced
import xgboost as xgb
from imblearn.over_sampling import SMOTE

# NLP
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
nltk.download('wordnet', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('omw-1.4', quiet=True)


print("All imports successful!")

All imports successful!


In [21]:
# Load dataset (ensure train.csv and test.csv are in your current directory)
train_path = 'train.csv'
test_path = 'test.csv'

train_df = pd.read_csv(train_path)
public_test_df = pd.read_csv(test_path)

# Basic exploration
print(f"Train size: {len(train_df)}")
print("Label Distribution:\n", train_df['label'].value_counts())
display(train_df.head())

Train size: 1302
Label Distribution:
 label
1    837
0    465
Name: count, dtype: int64


,id,content,label
0,1166,i think we should consider the actually the cl...,1
1,1127,Can you reset my password? My account password...,1
2,1240,im wondering if we need to no wait the test ca...,1
3,1853,"This code needs to be reviewed by tomorrow, do...",0
4,194,"This homework is taking forever, btw the new s...",0


In [23]:
# ── Standardize column names ───────────────────────────────────────────────────

# Based on your files, the text column is named 'content'
# We will normalize it to 'text' for consistency in the pipeline
text_col_mapping = {'content': 'text'}

# Rename 'content' to 'text' if it exists
train_df = train_df.rename(columns=text_col_mapping)
public_test_df = public_test_df.rename(columns=text_col_mapping)

# Ensure 'id' column exists (if not already present)
if 'id' not in train_df.columns:
    train_df['id'] = train_df.index
if 'id' not in public_test_df.columns:
    public_test_df['id'] = public_test_df.index

# ── Label Analysis (Training Set Only) ────────────────────────────────────────

print("Standardization complete.")
print(f"Final columns: {train_df.columns.tolist()}")

if 'label' in train_df.columns:
    counts = train_df['label'].value_counts()
    print("\nLabel distribution (train):")
    print(counts)

    # Calculate imbalance ratio if both classes 0 and 1 exist
    if 0 in counts and 1 in counts:
        ratio = counts[0] / counts[1]
        print(f"\nImbalance ratio (0/1): {ratio:.2f}")
    else:
        print("\nNote: Only one class found in 'label' column.")
else:
    print("\nWarning: 'label' column not found in train_df.")

# Note: public_test_df does not have a 'label' column, skipping its analysis.

Standardization complete.
Final columns: ['id', 'text', 'label']

Label distribution (train):
label
1    837
0    465
Name: count, dtype: int64

Imbalance ratio (0/1): 0.56


## Part 1: Data Balancing

You must implement and compare two methods:
1. **Basic (Required)**: Random Over-sampling.
2. **Advanced (Choose 1+)**: EDA, Back-translation, SMOTE, or Cost-Sensitive Learning.

In [24]:
# -----------------------------------------------------------------
# PHASE 1: Data Balancing
# -----------------------------------------------------------------

def perform_balancing(df, method='random'):
    """
    REQUIRED: method='random' (Random Over-sampling)
    OPTIONAL: method='advanced' (e.g., SMOTE or Cost-Sensitive hooks)
    """
    if method == 'random':
        # ── Step 1: split by class ─────────────────────────────────────────
        #TODO :Complete the function with the parameters
        #choose label 1 or 0
        # df_majority = df[df['label'] == ]
        # df_minority = df[df['label'] == ]
        df_majority = df[df['label'] == 0]
        df_minority = df[df['label'] == 1]

        # ── Step 2: upsample minority to match majority ────────────────────
        #TODO :Complete the function with the parameters
        # df_minority_upsampled = resample( )
        df_minority_upsampled = resample(
            df_minority,
            replace=True,
            n_samples=len(df_majority),
            random_state=42
        )

        # ── Step 3: combine and shuffle ────────────────────────────────────
        #TODO :Complete the function with the parameters
        # df_balanced = df_balanced.sample( )
        df_balanced = pd.concat([df_majority, df_minority_upsampled])
        df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)
        return df_balanced

    elif method == 'advanced':
        # [OPTIONAL PHASE 3]: Implement Advanced method
        # Hint: If using SMOTE, remember it must be applied AFTER TfidfVectorizer
        # because SMOTE works on numerical vectors, not raw text.
        return None

# Quick smoke-test
sample_balanced = perform_balancing(train_df, method='random') #choose your method
print("After Random Over-sampling:")
print(sample_balanced['label'].value_counts())

After Random Over-sampling:
label
1    465
0    465
Name: count, dtype: int64


## Part 2: Baseline Classifier (TF-IDF + SVM)

Establish a baseline. Use Macro-F1 as your primary metric.

In [ ]:
# -----------------------------------------------------------------
# PHASE  3: Text Preprocessing (Optional)
# -----------------------------------------------------------------

# ── Original template (kept for reference) ──
# """
# lemmatizer =
# stop_words =
# """
# def preprocess_text(text):
#
#     [OPTIONAL PHASE 3]: Preprocessing Function
#     Hint: Use .str.lower(), or apply a lambda function for NLTK PorterStemmer/WordNetLemmatizer.
#
# """
# # Apply to entire corpus
# train_df['text_clean'] = train_df['text'].apply(preprocess_text)
# public_test_df['text_clean'] = public_test_df['text'].apply(preprocess_text)
#
# print("Sample original :", train_df['text'].iloc[0])
# print("Sample cleaned  :", train_df['text_clean'].iloc[0])
# """

# ── Filled-in implementation ──
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))


def preprocess_text(text):
    """Lowercase, strip punctuation, drop stopwords, lemmatize."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = [
        lemmatizer.lemmatize(tok)
        for tok in text.split()
        if tok not in stop_words and len(tok) > 1
    ]
    return " ".join(tokens)


# Apply to entire corpus
train_df['text_clean'] = train_df['text'].apply(preprocess_text)
public_test_df['text_clean'] = public_test_df['text'].apply(preprocess_text)

print("Sample original :", train_df['text'].iloc[0])
print("Sample cleaned  :", train_df['text_clean'].iloc[0])


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PHASE 2  ─  Model
# ══════════════════════════════════════════════════════════════════════════════
def get_model(model_type='svm'):
    """
    Model Factory for Baseline and Advanced models.
    """
    if model_type == 'svm':
        # REQUIRED PHASE 2 Baseline
        # TODO : complete the function, you could add or change the parameters
        # return SVC(kernel='', probability=True, random_state=42)
        return SVC(kernel='linear', C=1.0, probability=True, random_state=42)

    elif model_type == 'advanced':
        # [OPTIONAL PHASE 3]
        # Hint: Initialize xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss').
        # Try tuning max_depth, n_estimators, and learning_rate.
        # return None
        return xgb.XGBClassifier(
            n_estimators=400,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.9,
            colsample_bytree=0.9,
            use_label_encoder=False,
            eval_metric='logloss',
            random_state=42,
            n_jobs=-1,
        )


In [ ]:

# --- Step 1: Internal Validation Setup ---
train_data, val_data = train_test_split(train_df, test_size=0.2, random_state=42, stratify=train_df['label'])


print(f"Train subset : {len(train_data)} samples")
print(f"Val   subset : {len(val_data)} samples")
print("Train label distribution:")
print(train_data['label'].value_counts())

In [ ]:
# TODO: Run your Baseline experiment
# 1. Balance data
# 2. Extract features (TF-IDF)
# 3. Train SVM
# 4. Evaluate using Macro-F1 and Classification Report

print("Running Baseline Experiment...")

# Define MODEL_TYPE locally for the baseline run
# This ensures this cell can be run independently for the baseline, even if kNNUqchf_Fic hasn't been run.
MODEL_TYPE = 'svm'
BALANCE_METHOD = 'random'

# 1. Balance data
balanced_train_data = perform_balancing(train_data.copy(), method=BALANCE_METHOD)
print(f"Balanced train size: {len(balanced_train_data)}  |  label dist:")
print(balanced_train_data['label'].value_counts())

# Pick the text column: prefer the preprocessed one when Phase 3 was run.
TEXT_COL = 'text_clean' if 'text_clean' in balanced_train_data.columns else 'text'

# 2. Extract features (TF-IDF)
# TODO : complete the function, you could add or change the parameters
# vectorizer = TfidfVectorizer(ngram_range=(), stop_words='')
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    stop_words='english',
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
)
#If applied text preprocessing:
#change balanced_train_data['text'] to balanced_train_data['text_clean']
# X_train = vectorizer.fit_transform(balanced_train_data['text'])
X_train = vectorizer.fit_transform(balanced_train_data[TEXT_COL])
y_train = balanced_train_data['label']

# 3. Train SVM
clf = get_model(MODEL_TYPE)
clf.fit(X_train, y_train)
print(f"Baseline model ({MODEL_TYPE}) trained on {X_train.shape[0]} balanced samples (column: {TEXT_COL}).")

# 4. Evaluate using Macro-F1 and Classification Report
# Prepare validation data
#If applied text preprocessing:
#change balanced_train_data['text'] to balanced_train_data['text_clean']
# X_val = vectorizer.transform(val_data['text'])
X_val = vectorizer.transform(val_data[TEXT_COL])
y_val = val_data['label']

# Make predictions
y_pred = clf.predict(X_val)
macro_f1 = f1_score(y_val, y_pred, average='macro')

print(f"=== Baseline: {MODEL_TYPE}| Balancing: {BALANCE_METHOD} ===")
print(classification_report(y_val, y_pred, target_names=['Incomplete(0)', 'Complete(1)']))
print(f"Macro-F1 Score: {macro_f1:.4f}")


## Part 3: Enhancement with K-Fold (Optional)

Improve your results with implement  K-FOLD Cross Validtaion if needed.

In [ ]:
# --- Step 1: Training Pipeline ---

# ══════════════════════════════════════════════════════════════════════════════
#  PHASE 3  ─  Stratified K-Fold Cross-Validation
#               Tests whether the simple split was just a lucky draw.
# ══════════════════════════════════════════════════════════════════════════════

def run_kfold_cv(df, model_type='svm', balance_method='random', n_splits=5, use_custom_preprocessing=False):
    """
    Run Stratified K-Fold CV and return fold scores + mean/std and all y_true/y_pred pairs.

    Parameters
    ----------
    df            : full training DataFrame
    model_type    : type of model to use ('svm' or 'advanced')
    balance_method: method for data balancing ('random' or 'advanced')
    n_splits      : number of folds (default 5)
    use_custom_preprocessing : bool, if True, uses the 'text_clean' column (after preprocess_text),
                                    else uses the original 'text' column.
    """
    text_col = 'text_clean' if (use_custom_preprocessing and 'text_clean' in df.columns) else 'text'

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    fold_scores, all_y_true, all_y_pred = [], [], []

    print(f"Running {n_splits}-Fold CV  |  model={model_type}  balance={balance_method}  text_col={text_col}")

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(df[text_col], df['label']), 1):
        fold_train = df.iloc[train_idx].reset_index(drop=True)
        fold_val   = df.iloc[val_idx].reset_index(drop=True)

        # 1. Balance only the train fold (never the val fold).
        if balance_method == 'random':
            fold_train_bal = perform_balancing(fold_train, method='random')
        else:
            fold_train_bal = fold_train.copy()

        # 2. Fit TF-IDF on the train fold only.
        vec = TfidfVectorizer(
            ngram_range=(1, 2),
            stop_words='english',
            min_df=2,
            max_df=0.95,
            sublinear_tf=True,
        )
        X_tr = vec.fit_transform(fold_train_bal[text_col])
        y_tr = fold_train_bal['label'].values

        # 3. (Advanced) SMOTE must run AFTER TF-IDF since it needs numeric vectors.
        if balance_method == 'advanced':
            sm = SMOTE(random_state=42, k_neighbors=5)
            X_tr, y_tr = sm.fit_resample(X_tr, y_tr)

        X_vl = vec.transform(fold_val[text_col])
        y_vl = fold_val['label'].values

        # 4. Train + predict + score.
        model = get_model(model_type)
        model.fit(X_tr, y_tr)
        y_hat = model.predict(X_vl)

        f1 = f1_score(y_vl, y_hat, average='macro')
        fold_scores.append(f1)
        all_y_true.extend(y_vl)
        all_y_pred.extend(y_hat)
        print(f"  Fold {fold_idx}/{n_splits}  Macro-F1 = {f1:.4f}")

    fold_scores = np.array(fold_scores)
    print(f"\nCV Macro-F1 = {fold_scores.mean():.4f}  ± {fold_scores.std():.4f}")
    print(classification_report(all_y_true, all_y_pred,
                                target_names=['Incomplete(0)', 'Complete(1)']))

    return {
        'fold_scores': fold_scores,
        'mean': fold_scores.mean(),
        'std': fold_scores.std(),
        'y_true': all_y_true,
        'y_pred': all_y_pred,
    }


# --- Step 2: Run the two configurations and compare ---
print("\n=== Baseline: SVM + Random Over-sampling ===")
cv_baseline = run_kfold_cv(
    train_df,
    model_type='svm',
    balance_method='random',
    n_splits=5,
    use_custom_preprocessing=('text_clean' in train_df.columns),
)

print("\n=== Advanced: XGBoost + SMOTE ===")
cv_advanced = run_kfold_cv(
    train_df,
    model_type='advanced',
    balance_method='advanced',
    n_splits=5,
    use_custom_preprocessing=('text_clean' in train_df.columns),
)

print("\n── Summary ──")
print(f"Baseline (SVM + RandomOS) : {cv_baseline['mean']:.4f} ± {cv_baseline['std']:.4f}")
print(f"Advanced (XGB + SMOTE)    : {cv_advanced['mean']:.4f} ± {cv_advanced['std']:.4f}")


## Part 4: Final Submission

Train on the full dataset using your best found configuration and generate `submission.csv`.

In [ ]:
# TODO: Train final model on the entire training set

# 1. Balance data using the best method found (e.g., 'random')
# Note: This will use the entire train_df, which has 'text_clean' if preprocessing was applied.
final_balanced_train_df = perform_balancing(train_df.copy(), method='random')
print(f"Full balanced training size: {len(final_balanced_train_df)}")
print(final_balanced_train_df['label'].value_counts())

# Pick text column: prefer text_clean when Phase 3 preprocessing ran.
FINAL_TEXT_COL = 'text_clean' if 'text_clean' in final_balanced_train_df.columns else 'text'

# 2. Extract features (TF-IDF) on the full balanced dataset using the preprocessed text
# Use the same TF-IDF parameters (ngram_range, stop_words) as determined during validation/K-Fold
# final_vectorizer = TfidfVectorizer() #TODO: fill in the parameters
final_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    stop_words='english',
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
)
# Ensure to use 'text_clean' column here for consistency with preprocessing if applied
# X_full_train = final_vectorizer.fit_transform(final_balanced_train_df['text'])
X_full_train = final_vectorizer.fit_transform(final_balanced_train_df[FINAL_TEXT_COL])
y_full_train = final_balanced_train_df['label']

# 3. Train final SVM model
# The model type and hyperparameters should be chosen based on K-Fold results if K-Fold applied
final_model = get_model('svm')
final_model.fit(X_full_train, y_full_train)
print(f"Final model (svm) trained on {X_full_train.shape[0]} samples using column '{FINAL_TEXT_COL}'.")


In [ ]:
# -----------------------------------------------------------------
# PHASE 4: Kaggle Submission
# -----------------------------------------------------------------

def generate_kaggle_submission(model, vectorizer, test_df, output_name='submission.csv'):
    """
    Generates the final CSV. Ensure test_df['text'] is processed the same way as training data.
    """
    # Ensure test data is preprocessed before vectorization, just like training data
    # If 'text_clean' exists from previous preprocessing, use it; otherwise, use 'text'
    # X_test = vectorizer.transform(test_df[text])   # original — `text` was an undefined name (NameError)
    text_col = 'text_clean' if 'text_clean' in test_df.columns else 'text'
    X_test = vectorizer.transform(test_df[text_col])
    test_predictions = model.predict(X_test)

    submission = pd.DataFrame({'id': test_df['id'], 'label': test_predictions})
    submission.to_csv(output_name, index=False)

    # Display first few rows of submission for verification
    print(f"Saved → {output_name}")
    print("Prediction distribution:")
    print(submission['label'].value_counts())
    display(submission.head(10))

    print(f"Success: {output_name} ready for Kaggle upload.")

# Final Step:
generate_kaggle_submission(final_model, final_vectorizer, public_test_df)


In [ ]:
# Download in Colab
try:
    from google.colab import files
    files.download('submission.csv')
    print("Download started!")
except ImportError:
    print("Not running in Colab — file saved locally as submission.csv")